# Genie / Agent Demo

This notebook shows how the curated semantic layer can be accessed through conversational AI.

## Example prompt

"Show me the top 5 campaigns by ROAS in the last 30 days and highlight any campaigns with average watch duration under 8 seconds."

## Conversational SQL query example

In [0]:
%sql
SELECT
  campaign_name,
  total_revenue,
  total_spend,
  roas,
  avg_view_duration_seconds
FROM adtech_demo.adtech_platform.vw_campaign_metrics
WHERE roas IS NOT NULL
ORDER BY roas DESC
LIMIT 5;

## Fallback path
- If Genie is unavailable, run the same query directly in Databricks SQL.
- Use the semantic view `adtech_demo.adtech_platform.vw_campaign_metrics` to keep the conversation on trusted metrics.

## Notes
- A full Genie implementation may require workspace-specific conversational AI configuration.
- This notebook is designed as a recipe for the agent use case, not the hosted Genie service itself.

---
# Budget Optimizer Agent

This agent applies **multi-step logical reasoning** over the trusted semantic layer (`vw_campaign_metrics`) to recommend budget reallocations.

### Reasoning Chain
1. **Retrieve** — Load campaign metrics from the governed view
2. **Classify** — Label each campaign as underperformer / performer / overperformer using ROAS thresholds
3. **Analyze** — Compute efficiency scores combining ROAS, CTR, and engagement
4. **Reallocate** — Shift budget from underperformers to overperformers proportionally
5. **Recommend** — Generate actionable, explainable recommendations

In [0]:
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
import json

@dataclass
class CampaignProfile:
    """Enriched campaign profile with classification and efficiency score."""
    campaign_name: str
    advertiser: str
    spend: float
    revenue: float
    roas: float
    ctr: float
    viewability_rate: float
    engaged_view_rate: float
    avg_view_duration: float
    classification: str = ""
    efficiency_score: float = 0.0


class BudgetOptimizerAgent:
    """
    A rule-based reasoning agent that optimizes ad budget allocation
    using trusted metrics from the semantic layer.
    """

    # ── Configurable thresholds ──
    ROAS_UNDERPERFORM = 1.0      # Below break-even
    ROAS_OVERPERFORM  = 1.15     # Strong return
    REALLOC_RATE      = 0.20     # Shift 20% of underperformer budget

    # Efficiency score weights
    W_ROAS       = 0.50
    W_CTR        = 0.25
    W_ENGAGEMENT = 0.25

    def __init__(self):
        self.campaigns: List[CampaignProfile] = []
        self.reasoning_log: List[str] = []

    # ── Step 1: Retrieve ──
    def load_data(self) -> "BudgetOptimizerAgent":
        """Fetch campaign metrics from the governed semantic view."""
        self._log("RETRIEVE", "Loading metrics from adtech_demo.adtech_platform.vw_campaign_metrics")
        df = spark.sql("SELECT * FROM adtech_demo.adtech_platform.vw_campaign_metrics")
        rows = df.collect()
        self.campaigns = [
            CampaignProfile(
                campaign_name=r["campaign_name"],
                advertiser=r["advertiser"],
                spend=float(r["total_spend"]),
                revenue=float(r["total_revenue"]),
                roas=float(r["roas"]),
                ctr=float(r["click_through_rate"]),
                viewability_rate=float(r["viewability_rate"]),
                engaged_view_rate=float(r["engaged_view_rate"]),
                avg_view_duration=float(r["avg_view_duration_seconds"]),
            )
            for r in rows
        ]
        self._log("RETRIEVE", f"Loaded {len(self.campaigns)} campaigns")
        return self

    # ── Step 2: Classify ──
    def classify_campaigns(self) -> "BudgetOptimizerAgent":
        """Label campaigns based on ROAS thresholds."""
        self._log("CLASSIFY", f"Thresholds — under: ROAS < {self.ROAS_UNDERPERFORM}, over: ROAS >= {self.ROAS_OVERPERFORM}")
        for c in self.campaigns:
            if c.roas < self.ROAS_UNDERPERFORM:
                c.classification = "underperformer"
            elif c.roas >= self.ROAS_OVERPERFORM:
                c.classification = "overperformer"
            else:
                c.classification = "performer"
            self._log("CLASSIFY", f"  {c.campaign_name}: ROAS={c.roas:.2f} → {c.classification}")
        return self

    # ── Step 3: Analyze ──
    def compute_efficiency(self) -> "BudgetOptimizerAgent":
        """Compute a weighted efficiency score combining ROAS, CTR, and engagement."""
        self._log("ANALYZE", f"Weights — ROAS: {self.W_ROAS}, CTR: {self.W_CTR}, Engagement: {self.W_ENGAGEMENT}")
        # Normalize each metric to [0, 1] range across campaigns
        max_roas = max(c.roas for c in self.campaigns) or 1
        max_ctr  = max(c.ctr for c in self.campaigns) or 1
        max_eng  = max(c.engaged_view_rate for c in self.campaigns) or 1

        for c in self.campaigns:
            c.efficiency_score = (
                self.W_ROAS       * (c.roas / max_roas) +
                self.W_CTR        * (c.ctr / max_ctr) +
                self.W_ENGAGEMENT * (c.engaged_view_rate / max_eng)
            )
            self._log("ANALYZE", f"  {c.campaign_name}: efficiency={c.efficiency_score:.3f}")
        return self

    # ── Step 4: Reallocate ──
    def optimize_budget(self) -> Dict:
        """Shift budget from underperformers to overperformers."""
        underperformers = [c for c in self.campaigns if c.classification == "underperformer"]
        overperformers  = sorted(
            [c for c in self.campaigns if c.classification == "overperformer"],
            key=lambda c: c.efficiency_score, reverse=True
        )

        freed_budget = sum(c.spend * self.REALLOC_RATE for c in underperformers)
        self._log("REALLOCATE", f"Freed ${freed_budget:.2f} from {len(underperformers)} underperformer(s)")

        allocations = []
        if overperformers and freed_budget > 0:
            total_eff = sum(c.efficiency_score for c in overperformers)
            for c in overperformers:
                share = (c.efficiency_score / total_eff) * freed_budget if total_eff else 0
                allocations.append({
                    "campaign": c.campaign_name,
                    "current_spend": round(c.spend, 2),
                    "additional_budget": round(share, 2),
                    "new_spend": round(c.spend + share, 2),
                    "efficiency_score": round(c.efficiency_score, 3),
                })
                self._log("REALLOCATE", f"  +${share:.2f} → {c.campaign_name} (eff={c.efficiency_score:.3f})")

        cuts = [{
            "campaign": c.campaign_name,
            "current_spend": round(c.spend, 2),
            "budget_cut": round(c.spend * self.REALLOC_RATE, 2),
            "new_spend": round(c.spend * (1 - self.REALLOC_RATE), 2),
            "roas": round(c.roas, 2),
        } for c in underperformers]

        return {
            "freed_budget": round(freed_budget, 2),
            "cuts": cuts,
            "allocations": allocations,
        }

    # ── Step 5: Recommend ──
    def generate_recommendations(self, result: Dict) -> str:
        """Produce a human-readable recommendation summary."""
        lines = ["═" * 60, "  BUDGET OPTIMIZER AGENT — RECOMMENDATIONS", "═" * 60, ""]

        if not result["cuts"]:
            lines.append("✅ All campaigns are performing at or above target ROAS.")
            lines.append("   No budget reallocation needed.")
        else:
            lines.append(f"💰 Total freed budget: ${result['freed_budget']:.2f}")
            lines.append("")
            lines.append("🔻 REDUCE spend on underperformers:")
            for cut in result["cuts"]:
                lines.append(f"   • {cut['campaign']}: ${cut['current_spend']:.2f} → ${cut['new_spend']:.2f}  (ROAS: {cut['roas']})")

            lines.append("")
            lines.append("🔺 INCREASE spend on top performers:")
            for alloc in result["allocations"]:
                lines.append(f"   • {alloc['campaign']}: ${alloc['current_spend']:.2f} → ${alloc['new_spend']:.2f}  (+${alloc['additional_budget']:.2f}, eff: {alloc['efficiency_score']})")

        lines.append("")
        lines.append("─" * 60)
        lines.append("REASONING TRACE:")
        lines.append("─" * 60)
        for entry in self.reasoning_log:
            lines.append(entry)

        return "\n".join(lines)

    # ── Run full chain ──
    def run(self) -> str:
        """Execute the full reasoning chain and return recommendations."""
        self.load_data()
        self.classify_campaigns()
        self.compute_efficiency()
        result = self.optimize_budget()
        return self.generate_recommendations(result)

    def _log(self, step: str, message: str):
        self.reasoning_log.append(f"[{step}] {message}")

print("✅ BudgetOptimizerAgent defined")

In [0]:
agent = BudgetOptimizerAgent()
recommendations = agent.run()
print(recommendations)

## Agent Reasoning Actions Demonstrated

| Step | Action | Reasoning Type |
| --- | --- | --- |
| Retrieve | Load from governed semantic view | Data access via trusted layer |
| Classify | Label campaigns by ROAS thresholds | Threshold-based classification |
| Analyze | Weighted efficiency scoring (ROAS + CTR + engagement) | Multi-factor ranking |
| Reallocate | Proportional budget shift from underperformers | Optimization under constraints |
| Recommend | Natural-language summary with full trace | Explainable AI / auditability |

> **Key insight**: The agent's reasoning is only as trustworthy as the data it consumes.  
> By building on the governed semantic layer (`vw_campaign_metrics`), every recommendation is traceable back to quality-checked, lineage-tracked metrics.